## GollumFit Examples

### Fast MC

In [ ]:
"""
FastMC Generation Example

This script demonstrates how to generate FastMC, which is a compressed
representation of Monte Carlo that significantly improves fitting efficiency.

Example command:
    nohup python generate_fastMC.py > generate_fastMC.log 2>&1 &
"""

import GollumFitPy as gf
import numpy as np
import os
import sys
sys.stdout.flush()

In [ ]:
#####################################################################################
# Configure Data Paths - Set paths for cross section splines
#####################################################################################
datapaths = gf.DataPaths()

gollum_dir = "GollumFit/GollumFit"

datapaths.neutrino_cc_xs_spline_path             = gollum_dir + "/resources/Splines/CrossSections/sigma_nu_CC_iso.fits"
datapaths.antineutrino_cc_xs_spline_path         = gollum_dir + "/resources/Splines/CrossSections/sigma_nubar_CC_iso.fits"
datapaths.neutrino_nc_xs_spline_path             = gollum_dir + "/resources/Splines/CrossSections/sigma_nu_NC_iso.fits"
datapaths.antineutrino_nc_xs_spline_path         = gollum_dir + "/resources/Splines/CrossSections/sigma_nubar_NC_iso.fits"
datapaths.diff_neutrino_cc_xs_spline_path        = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nu_CC_iso.fits"
datapaths.diff_antineutrino_cc_xs_spline_path    = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nubar_CC_iso.fits"
datapaths.diff_neutrino_nc_xs_spline_path        = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nu_NC_iso.fits"
datapaths.diff_antineutrino_nc_xs_spline_path    = gollum_dir + "/resources/Splines/CrossSections/dsdxdy_nubar_CC_iso.fits"
datapaths.mc_path                                = gollum_dir + "/monte_carlo/"
datapaths.domeff_spline_path                     = gollum_dir + "/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.holeice_spline_path                    = gollum_dir + "/resources/Splines/HoleIceSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.attenuation_spline_path                = gollum_dir + "/resources/Splines/AttenuationSplines/new_ddmnodeis"
datapaths.ice_gradient_spline_path               = gollum_dir + "/resources/Splines/IceGradientsSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.atmospheric_density_spline_path        = gollum_dir + "/resources/Splines/AtmosphericZenithVariationSplines/atm_density_1s.fits"
datapaths.atmospheric_kaonlosses_spline_path     = gollum_dir + "/resources/Splines/AtmosphericKaonLossesSplines/kaon_loses_1s.fits"


In [ ]:
#####################################################################################
# Configure Flux Files - Load atmospheric, prompt, and astrophysical flux files
#####################################################################################
datapaths.conventional_nusquids_atmospheric_file = gollum_dir + "/examples/fluxes/atmospheric.hdf5"
datapaths.prompt_nusquids_atmospheric_file       = gollum_dir + "/examples/fluxes/prompt_atmospheric.hdf5"
datapaths.astro_nusquids_file                    = gollum_dir + "/examples/fluxes/astro.hdf5"

# Hadronic and cosmic ray correction splines (necessary for flux nuisance parameters)
hadronlist = ["he_K+", "he_K-", "vhe1_pi+", "vhe1_pi-", "vhe3_K+", "vhe3_K-", 
              "vhe3_pi+", "vhe3_pi-", "vhe3_p", "vhe3_n"]
crlist = ["GSF_1", "GSF_2", "GSF_3", "GSF_4", "GSF_5", "GSF_6"]

datapaths.hadronic_spline_path   = gollum_dir + "/examples/fluxes"
datapaths.cosmic_ray_spline_path = gollum_dir + "/examples/fluxes"

In [ ]:
#####################################################################################
# Set Steering Parameters - Configure analysis binning and settings
# NOTE: Binning choices affect FastMC compression and must match analysis configuration
#####################################################################################
steering_params = gf.SteeringParams()
steering_params.minFitEnergy                    = 300
steering_params.maxFitEnergy                    = 1e5
steering_params.logEbinEdge                     = np.log10(300)
steering_params.logEbinWidth                    = (np.log10(1e5) - np.log10(300)) / 24
steering_params.minCosth                        = -1.0
steering_params.maxCosth                        = 0.0
steering_params.cosThbinEdge                    = 0.0
steering_params.cosThbinWidth                   = 0.05
steering_params.selectionStart                  = 0.99
steering_params.ice_gradient_filename           = ["Amp_0", "Amp_1", "Amp_2", "Amp_3", "Amp_4", 
                                                   "Phs_1", "Phs_2", "Phs_3", "Phs_4"]
steering_params.active_hadronic_parameters      = hadronlist
steering_params.active_cosmicray_parameters     = crlist

# Livetime for the corresponding Monte Carlo
years = 10.669
steering_params.fullLivetime                    = years * 365 * 24 * 60 * 60.
steering_params.simToLoad                       = "BDT_Split_HE"
steering_params.energyName                      = "DnnEnergy"
steering_params.model_label                     = ""  # Can be used for uniquely-labelled flux files

In [ ]:
#####################################################################################
# Construct and Write FastMC
#####################################################################################
gollumfit = gf.GollumFit(datapaths, steering_params)

# Compression parameter: smaller values = higher compression but potential accuracy loss
metascaling = 0.25
gollumfit.ConstructFastMode(metascaling)

# Write to file
gollumfit.WriteCompact("compact.fastmc")

print("Done generating FastMC.")

---

### Generating Fake Data

<!-- Generating Fake data with non-nominal values i.e. the values do not follow the exact distribution that was in the original icecube example, instead I'll be changing the means to some other value to test if GollumFit can recover the samples 

The code here is based on the generate_fakedata.py file in the fitting_to_null example  -->

In [1]:
"""
Generate Null Pseudo-Data Example

This script generates null pseudo-data (expectation with nominal nuisance parameters)
for use in fitting examples.

Example command:
    nohup time python ./generate_fakedata.py > generate_fakedata.log 2>&1 &
"""

import GollumFitPy as gf
import numpy as np
import os
import sys
import h5py
from collections import OrderedDict
import scipy.stats as stats


In [2]:
#####################################################################################
# Define Nuisance Parameters
# Format: [vary_flag, prior_type, center, width, lower_bound, upper_bound]
#####################################################################################

syst_dict     = OrderedDict({ 
    'convNorm'                  : [ True, 'Gaussian',      1.,   0.2,                   0.1,                   3. ], 
    'zenithCorrection'          : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'kaonLosses'                : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'hadronicHEkp'              : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicHEkm'              : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE1pip'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE1pim'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3kp'            : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3km'            : [ True, 'Gaussian',      0.,    1.,                  -1.5,                   2. ], 
    'hadronicVHE3pip'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3pim'           : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3p'             : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3n'             : [ True, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'cosmicRay1'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay2'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay3'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay4'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay5'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay6'                : [ True, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'icegrad0'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad1'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad2'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad3'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad4'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad5'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad6'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad7'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad8'                  : [ True, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'domEfficiency'             : [ True, 'Gaussian',    1.27, 0.123,                 1.234,                1.346 ], 
    'holeiceForward'            : [ True, 'Gaussian',     -1.,   10.,                 -5.35,                 1.85 ], 
    'astroNorm'                 : [ True, 'Gaussian', 4.72/6.,  0.36,                    0.,                   3. ], 
    'astroDeltaGamma'           : [ True, 'Gaussian',      0.,  0.36,                   -2.,                   2. ], 
    'astroDeltaGammaSec'        : [ True, 'Gaussian',      0.,  0.36,                   -2.,                   2. ], 
    'nuxs'                      : [ True, 'Gaussian',      1.,   0.1,                 0.824,                1.176 ], 
    'nubarxs'                   : [ True, 'Gaussian',      1.,   0.1,                 0.824,                1.176 ], 
    'astroPivot'                : [ True,  'Uniform',      5.,    1.,                    4.,                   6. ], 
    'promptNorm'                : [ True, 'Gaussian',      1.,    1.,                    0.,                   3. ],
    'NeutrinoAntineutrinoRatio' : [ True, 'Gaussian',      1.,    1.,                    0.,                   2. ],
})

In [ ]:
#####################################################################################
# set paths to relevant splines and fastMC
#####################################################################################
datapaths = gf.DataPaths()

gollum_dir = "GollumFit/GollumFit"

# leave these three empty with "" if you dont want to use the splines
datapaths.domeff_spline_path      = gollum_dir + "/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.holeice_spline_path     = gollum_dir + "/resources/Splines/HoleIceSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.attenuation_spline_path = gollum_dir + "/resources/Splines/AttenuationSplines/new_ddmnodeis"

# do NOT leave this empty though
datapaths.compact_file_path       = "Data/GollumFit_Data/compact.fastmc"

In [4]:
#####################################################################################
# steering params to set the binning 
#####################################################################################
steering_params = gf.SteeringParams()
steering_params.minFitEnergy                    = 300
steering_params.maxFitEnergy                    = 1e5
steering_params.logEbinEdge                     = np.log10(300)
steering_params.logEbinWidth                    = (np.log10(1e5)-np.log10(300))/24
steering_params.minCosth                        = -1
steering_params.maxCosth                        = 0.
steering_params.cosThbinEdge                    = 0.0
steering_params.cosThbinWidth                   = 0.05
steering_params.selectionStart                  = float("DnnEnergy_0.99".split("_")[1])
steering_params.evalThreads                     = 1

In [ ]:
#####################################################################################
# declare gollumfit object
#####################################################################################
gollumfit = gf.GollumFit(datapaths,steering_params)

reset_steering: 1
reset_data: 1
Constructing DiffuseWeightMaker
Loading compact data
Checking file from path Data/GollumFit_Data/compact.fastmc
Making sim hist


In [ ]:
#####################################################################################
# Input the nuisance parameter values (at null) and generate and save the expectation
# events. We can then use this as "fake data"
# In general, we can use any combination of nuisance parameter values to see what the 
# expecation would be, and fit to it. This is useful for doing mismodelling tests, for
# example. 
# And when we are ready for the analysis, the real data can be input with the same
# format as the fake data, and the fit to real data can be seamlessly performed. 
#####################################################################################

fitparams = gf.FitParameters()

# set the nuisance parameter values
for sname in syst_dict.keys():
    if sname: 
        exec('fitparams.'+sname+' = syst_dict[\"'+sname+'\"][2]')

null_dist = gollumfit.GetExpectationEvents(fitparams)

print('saving to npz file.')
realization = "Data/GollumFit_Data/generated_data.npz"
np.savez(realization, realization=null_dist)


print("Done. Data saved to "+realization)

saving to npz file.
Done. Data saved to Data/GollumFit_Data/modified_fake_data.npz


In [ ]:
# To load the .npz file, use the np.load function along with the 
# name of the array that you had given

np.load("Data/GollumFit_Data/generated_data.npz")['realization']

array([[3.63081573e+02, 1.71265996e+00, 1.00000000e+00, 2.09718958e-02],
       [2.86473907e+02, 2.37640357e+00, 1.00000000e+00, 1.54020471e-02],
       [4.33857239e+02, 2.11349440e+00, 1.00000000e+00, 1.51350415e-02],
       ...,
       [7.51215271e+02, 1.73532403e+00, 1.00000000e+00, 9.42844971e-03],
       [3.46167206e+02, 2.65284657e+00, 0.00000000e+00, 4.04292196e-03],
       [1.43075760e+02, 2.85794544e+00, 1.00000000e+00, 3.28508235e-03]],
      shape=(13061947, 4))

---

### Performing likelihood minimization 

In [ ]:
"""
Likelihood Minimization Example - Fitting to Null Hypothesis

This script demonstrates GollumFit's core functionality: fitting Monte Carlo to
data using likelihood minimization. We fit to null pseudo-data starting from
random initial parameter values.

Example command:
    nohup time python ./generate_likelihood.py > generate_likelihood.log 2>&1 &
"""

import GollumFitPy as gf
import numpy as np
import os
import sys
import subprocess
import scipy.stats as stats

print('Starting generate_likelihood.py.')

In [ ]:
#####################################################################################
# Define Nuisance Parameters (False: vary in fit, True: do NOT vary in fit)
# Format: [vary_flag, prior_type, center, width, lower_bound, upper_bound]
#####################################################################################
syst_dict     = { 
    'convNorm'                  : [ False, 'Gaussian',      1.,   0.2,                   0.1,                   3. ], 
    'zenithCorrection'          : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'kaonLosses'                : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'hadronicHEkp'              : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicHEkm'              : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE1pip'           : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE1pim'           : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3kp'            : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3km'            : [ False, 'Gaussian',      0.,    1.,                  -1.5,                   2. ], 
    'hadronicVHE3pip'           : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3pim'           : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3p'             : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'hadronicVHE3n'             : [ False, 'Gaussian',      0.,    1.,                   -2.,                   2. ], 
    'cosmicRay1'                : [ False, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay2'                : [ False, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay3'                : [ False, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay4'                : [ False, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay5'                : [ False, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'cosmicRay6'                : [ False, 'Gaussian',      0.,    1.,                   -4.,                   4. ], 
    'icegrad0'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad1'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad2'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad3'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad4'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad5'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad6'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad7'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'icegrad8'                  : [ False, 'Gaussian',      0.,    1.,                   -3.,                   3. ], 
    'domEfficiency'             : [ False, 'Gaussian',    1.27, 0.123,                 1.234,                1.346 ], 
    'holeiceForward'            : [ False, 'Gaussian',     -1.,   10.,                 -5.35,                 1.85 ], 
    'astroNorm'                 : [ False, 'Gaussian', 4.72/6.,  0.36,                    0.,                   3. ], 
    'astroDeltaGamma'           : [ False, 'Gaussian',      0.,  0.36,                   -2.,                   2. ], 
    'astroDeltaGammaSec'        : [ False, 'Gaussian',      0.,  0.36,                   -2.,                   2. ], 
    'nuxs'                      : [ False, 'Gaussian',      1.,   0.1,                 0.824,                1.176 ], 
    'nubarxs'                   : [ False, 'Gaussian',      1.,   0.1,                 0.824,                1.176 ], 
    'astroPivot'                : [ False,  'Uniform',      5.,    1.,                    4.,                   6. ], 
    'promptNorm'                : [ False, 'Gaussian',      1.,    1.,                    0.,                   3. ],
    'NeutrinoAntineutrinoRatio' : [ False, 'Gaussian',      1.,    1.,                    0.,                   2. ],
}

In [ ]:
#####################################################################################
# Define Random Sampling Function
# Helper function to sample random parameter values from prior distributions
#####################################################################################
def throw(syst):
    """Sample random value from prior distribution."""
    if syst[1] == 'Gaussian':
        # Truncated normal distribution
        val = stats.truncnorm(
            (syst[4] - syst[2]) / syst[3],  # Lower bound (standardized)
            (syst[5] - syst[2]) / syst[3],  # Upper bound (standardized)
            syst[2],  # Mean
            syst[3]   # Std dev
        ).rvs(1)
        return val[0]
    else:
        # Uniform distribution
        return np.random.uniform(syst[4], syst[5])

In [ ]:
#####################################################################################
# Initialize Fit Parameter Objects
# Create objects to manage fit configuration
#####################################################################################

fitparams_flag  = gf.FitParametersFlag()  # Which parameters to vary
fitparams_bound = gf.FitParametersBound()  # Parameter bounds
priors          = gf.Priors()              # Prior distributions
seed_fitparams  = gf.FitParameters()       # Initial values

In [ ]:
#####################################################################################
# Set Priors and Random Initial Values
# Configure priors and randomly initialize starting parameter values
#####################################################################################
np.random.seed(100)  # For reproducibility
print('Initializing with the following randomly-seeded nuisance params:')

for sname in syst_dict.keys():
    # Set flags and bounds
    exec(f'fitparams_flag.{sname} = syst_dict["{sname}"][0]')
    exec(f'fitparams_bound.{sname}Min = syst_dict["{sname}"][4]')
    exec(f'fitparams_bound.{sname}Max = syst_dict["{sname}"][5]')
    
    # Set priors
    if syst_dict[sname][1] == 'Gaussian':
        exec(f'priors.{sname}Center = syst_dict["{sname}"][2]')
        exec(f'priors.{sname}Width  = syst_dict["{sname}"][3]')
    else:
        exec(f'priors.{sname}Min = syst_dict["{sname}"][4]')
        exec(f'priors.{sname}Max = syst_dict["{sname}"][5]')
    
    # Randomly initialize
    thrown_val = throw(syst_dict[sname])
    exec(f'seed_fitparams.{sname} = thrown_val')
    print(f'{sname}: {thrown_val}')

In [ ]:
#####################################################################################
# Set Correlations and Paths
# Load correlation matrices for ice gradients and flux parameters
#####################################################################################
gollum_dir = "GollumFit/GollumFit"

# Set correlations (required for fitting/minimization, not for likelihood evaluation)
iceg_corr = np.load(gollum_dir + '/resources/correlation_matrices/icegrad_correlations.npy')
flux_corr = np.load(gollum_dir + '/resources/correlation_matrices/flux_correlations_new_ddmnodeis.npy')
for idx, val in np.ndenumerate(iceg_corr):
    priors.SetIceGradientsCorr(idx[0], idx[1], val)
for idx, val in np.ndenumerate(flux_corr):
    priors.SetFluxCorr(idx[0], idx[1], val)

datapaths = gf.DataPaths()

# set these next three to "" if you dont want to use the splines
datapaths.domeff_spline_path      = gollum_dir + "/resources/Splines/DOMEffSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.holeice_spline_path     = gollum_dir + "/resources/Splines/HoleIceSplines/new_ddmnodeis/BDT/DnnEnergy_0.99"
datapaths.attenuation_spline_path = gollum_dir + "/resources/Splines/AttenuationSplines/new_ddmnodeis"

# data path corresponding to the .fastmc file
datapaths.compact_file_path       = gollum_dir + "/examples/FastMC/compact.fastmc"

In [ ]:
#####################################################################################
# Configure Steering Parameters
# Set binning and convergence criteria (must match FastMC binning)
#####################################################################################
edges = np.logspace(np.log10(300), np.log10(1e5), 25)
steering_params                = gf.SteeringParams()
steering_params.minFitEnergy   = edges[0]
steering_params.maxFitEnergy   = edges[-1]
steering_params.logEbinEdge    = np.log10(edges[0])
steering_params.logEbinWidth   = np.log10(edges[1]) - np.log10(edges[0])
steering_params.minCosth       = -1.0
steering_params.maxCosth       = 0.0
steering_params.cosThbinEdge   = 0.0
steering_params.cosThbinWidth  = 0.05
steering_params.selectionStart = float("DnnEnergy_0.99".split("_")[1])
steering_params.evalThreads    = 1

# Convergence criteria (tight tolerances for accurate minimization)
steering_params.change_tol     = 1.e-20#####################################################################################
# Load Data and Configure Fit
# Create GollumFit object and load pseudo-data
#####################################################################################
gollumfit = gf.GollumFit(datapaths, steering_params)

#####################################################################################
# declare the fake data location and load it
#####################################################################################
realization  = "nullexpectation.npz"
total_data = gollumfit.SetData(np.load(realization)["realization"])
steering_params.grad_tol       = 1.e-20
steering_params.uncertaintyModSigmaOverMu = 0.0

In [ ]:
#####################################################################################
# Load Data and Configure Fit
# Create GollumFit object and load pseudo-data
#####################################################################################
gollumfit = gf.GollumFit(datapaths, steering_params)

#####################################################################################
# declare the fake data location and load it
#####################################################################################
realization  = "Data/GollumFit_Data/generated_data.npz"
total_data = gollumfit.SetData(np.load(realization)["realization"])

#####################################################################################
# feed the flags, bounds, priors, on the nuisance parameters into gollumfit
#####################################################################################
gollumfit.SetFitParametersFlag(fitparams_flag)
gollumfit.SetFitParametersBound(fitparams_bound)
gollumfit.SetFitParametersPriors(priors)
gollumfit.SetFitParametersSeed([seed_fitparams])
gollumfit.ConstructLikelihoodProblem()

In [ ]:
#####################################################################################
# perform the minimization
#####################################################################################
print("Starting minimization...")
min_llh = gollumfit.MinLLH()

#####################################################################################
# results: print the best fit nuisance parameters, likelihood, and the number of LLH evaluations
#####################################################################################
systematics = ""
for sname in syst_dict.keys() :
    exec('print(\"'+sname+'\",min_llh.params.'+sname+')')
    exec('systematics += str(min_llh.params.'+sname+')+\" \"')

print('llh:',min_llh.likelihood)
print('nEval: '+str(min_llh.nEval))

print("Completed successfully. Bye!")